# 대시보드용

In [ ]:
# ============================================================
# 정상기업 부실 확률 변화율 분석
# 2021_2024_PD_데이터.csv 기반 (변화율 → 차이값으로 수정)
# ============================================================

import pandas as pd
import numpy as np
import os

PD_SAVE_DIR = r'15번. 우수모델 데이터'
PD_PATH     = os.path.join(PD_SAVE_DIR, "2021_2024_PD_데이터.csv")

YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
ID_COL       = "사업자등록번호"
THRESHOLD    = 0.44
TARGET_YEARS = [2021, 2022, 2023, 2024]
SAVE_DIR     = r"21번. 기업 PD 변화율"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── CSV 로드 ──────────────────────────────────────────────
pd_df = pd.read_csv(PD_PATH, encoding="utf-8-sig")
pd_df[YEAR_COL] = pd_df[YEAR_COL].astype(int)

print("=" * 65)
print(f"PD 데이터 로드 완료: {len(pd_df)}행")
print(f"연도 범위: {pd_df[YEAR_COL].min()} ~ {pd_df[YEAR_COL].max()}")
print(f"기업 수: {pd_df[ID_COL].nunique()}개")
print("=" * 65)


# ============================================================
# 대상 기업 필터링
# ============================================================

normal_2023 = pd_df[
    (pd_df[YEAR_COL] == 2023) &
    (pd_df["y_true"] == 0)
][[COMPANY_COL, ID_COL]].drop_duplicates()

target_ids = normal_2023[ID_COL].tolist()
print(f"2023년 정상 기업 수: {len(normal_2023)}개")

analysis_data = pd_df[
    pd_df[ID_COL].isin(target_ids) &
    pd_df[YEAR_COL].isin(TARGET_YEARS)
].copy()

year_count = (
    analysis_data.groupby(ID_COL)[YEAR_COL]
    .nunique()
    .reset_index()
    .rename(columns={YEAR_COL: "year_count"})
)
complete_ids  = year_count[year_count["year_count"] == 4][ID_COL].tolist()
analysis_data = analysis_data[analysis_data[ID_COL].isin(complete_ids)].copy()

print(f"2021~2024 모두 존재하는 기업 수: {len(complete_ids)}개")
print(f"분석 대상 행 수: {len(analysis_data)}행")
print("=" * 65)


# ============================================================
# 연도별 피벗
# ============================================================

pivot = analysis_data.pivot_table(
    index=[ID_COL, COMPANY_COL],
    columns=YEAR_COL,
    values="y_prob"
).reset_index()
pivot.columns.name = None
pivot = pivot.rename(columns={
    2021: "prob_2021",
    2022: "prob_2022",
    2023: "prob_2023",
    2024: "prob_2024",
})
pivot = pivot.dropna(
    subset=["prob_2021", "prob_2022", "prob_2023", "prob_2024"]
).reset_index(drop=True)


# ============================================================
# ★ 차이값 계산 (현재 - 이전, 단위: 확률 그대로)
# 양수 = 부실확률 상승, 음수 = 부실확률 하락
# ============================================================

pivot["diff_2021_2022"] = (pivot["prob_2022"] - pivot["prob_2021"]).round(4)
pivot["diff_2022_2023"] = (pivot["prob_2023"] - pivot["prob_2022"]).round(4)
pivot["diff_2023_2024"] = (pivot["prob_2024"] - pivot["prob_2023"]).round(4)


# ============================================================
# 예측 라벨
# ============================================================

for yr in TARGET_YEARS:
    pivot[f"pred_label_{yr}"] = (pivot[f"prob_{yr}"] >= THRESHOLD).astype(int)


# ============================================================
# 위험 신호 분류 (차이값 기준으로 재정의)
# 가속: diff 자체가 점점 커지는 것 (diff_22 < diff_23 < diff_24)
# ============================================================

def risk_signal(row):
    p24  = row["prob_2024"]
    d_22 = row["diff_2021_2022"]   # 2021→2022 차이
    d_23 = row["diff_2022_2023"]   # 2022→2023 차이
    d_24 = row["diff_2023_2024"]   # 2023→2024 차이

    # 3년 연속 상승 + 가속 (diff 자체도 점점 커짐)
    accel_3 = (
        d_22 > 0 and d_23 > 0 and d_24 > 0 and
        d_22 < d_23 < d_24
    )
    # 3년 연속 상승
    rising_3 = d_22 > 0 and d_23 > 0 and d_24 > 0

    # 최근 2년 상승 + 가속
    accel_2  = d_23 > 0 and d_24 > 0 and d_23 < d_24

    # 최근 2년 상승
    rising_2 = d_23 > 0 and d_24 > 0

    if accel_3:
        if p24 >= THRESHOLD:
            return "🔴 위험 (가속+임계초과)"
        elif p24 >= THRESHOLD * 0.7:
            return "🟠 주의 (가속+임계근접)"
        else:
            return "🟡 관찰 (가속)"

    if rising_3:
        if p24 >= THRESHOLD:
            return "🔴 위험 (3년연속상승+임계초과)"
        elif p24 >= THRESHOLD * 0.7:
            return "🟠 주의 (3년연속상승+임계근접)"
        else:
            return "🟡 관찰 (3년연속상승)"

    if accel_2:
        if p24 >= THRESHOLD:
            return "🔴 위험 (최근가속+임계초과)"
        elif p24 >= THRESHOLD * 0.7:
            return "🟠 주의 (최근가속+임계근접)"
        else:
            return "🟡 관찰 (최근가속)"

    if rising_2:
        if p24 >= THRESHOLD:
            return "🔴 위험 (최근2년상승+임계초과)"
        elif p24 >= THRESHOLD * 0.7:
            return "🟠 주의 (최근2년상승+임계근접)"
        else:
            return "🟡 관찰 (최근2년상승)"

    # 최근 구간만 큰 폭 상승 (0.05 이상)
    if d_24 > 0.05:
        if p24 >= THRESHOLD:
            return "🔴 위험 (최근급등+임계초과)"
        else:
            return "🟠 주의 (최근급등)"

    # 추세 무관 임계값 초과
    if p24 >= THRESHOLD:
        return "🔴 위험 (임계초과)"

    # 안정
    if p24 < THRESHOLD * 0.5:
        return "🟢 안정"

    return "⚪ 보통"


pivot["_risk_signal"] = pivot.apply(risk_signal, axis=1)


# ============================================================
# 순위 기반 위험 점수 (0~100)
# 최근 연도일수록 가중치 높게
# ============================================================

pivot["rank_diff_22"] = pivot["diff_2021_2022"].rank(pct=True) * 100
pivot["rank_diff_23"] = pivot["diff_2022_2023"].rank(pct=True) * 100
pivot["rank_diff_24"] = pivot["diff_2023_2024"].rank(pct=True) * 100
pivot["rank_prob_24"] = pivot["prob_2024"].rank(pct=True) * 100

# 추세 점수 (가중 합산) + 현재 절대 수준 반영
pivot["risk_score"] = (
    (
        pivot["rank_diff_22"] * 0.2 +
        pivot["rank_diff_23"] * 0.3 +
        pivot["rank_diff_24"] * 0.5
    ) * 0.5 +
    pivot["rank_prob_24"] * 0.5
).round(2)


# ============================================================
# 컬럼 정리
# ============================================================

result_df = pivot[[
    ID_COL, COMPANY_COL,
    "prob_2021", "prob_2022", "prob_2023", "prob_2024",
    "pred_label_2021", "pred_label_2022",
    "pred_label_2023", "pred_label_2024",
    "diff_2021_2022", "diff_2022_2023", "diff_2023_2024",
    "risk_score",
    "_risk_signal",
]].copy()

# ── prob 소수점 ───────────────────────────────────────────
for col in ["prob_2021", "prob_2022", "prob_2023", "prob_2024"]:
    result_df[col] = result_df[col].round(4)

# ── risk_score 소수점 ─────────────────────────────────────
result_df["risk_score"] = result_df["risk_score"].round(2)


# ============================================================
# 정렬 (risk_score 내림차순)
# ============================================================

result_df = result_df.sort_values(
    "risk_score", ascending=False
).reset_index(drop=True)


# ============================================================
# 저장 (_risk_signal 제외)
# ============================================================

save_cols        = [c for c in result_df.columns if c != "_risk_signal"]
change_save_path = os.path.join(SAVE_DIR, "2022-2024 정상기업_부실확률_차이값.csv")
result_df[save_cols].to_csv(
    change_save_path, index=False, encoding="utf-8-sig"
)


# ============================================================
# 요약 출력
# ============================================================

print(f"\n{'='*65}")
print(f"정상기업 부실 확률 차이값 분석 완료")
print(f"  → {change_save_path}  ({len(result_df)}행)")

print(f"\n  [위험 신호 분포]")
for sig, cnt in result_df["_risk_signal"].value_counts().items():
    print(f"    {sig} : {cnt}개")

print(f"\n  [상위 10개 (risk_score 높은 순)]")
print(result_df[[
    COMPANY_COL,
    "prob_2021", "prob_2022", "prob_2023", "prob_2024",
    "diff_2021_2022", "diff_2022_2023", "diff_2023_2024",
    "risk_score", "_risk_signal"
]].head(10).to_string(index=False))

print("=" * 65)

PD 데이터 로드 완료: 35180행
연도 범위: 2014 ~ 2024
기업 수: 5684개
2023년 정상 기업 수: 3829개
2014~2024 모두 존재하는 기업 수: 1177개
분석 대상 행 수: 12947행

정상기업 부실 확률 차이값 분석 완료
  → 21번. 기업 PD 변화율\2015-2024 정상기업_부실확률_차이값.csv  (1177행)

  [위험 신호 분포]
    🟢 안정 : 678개
    🟡 관찰 (최근가속) : 147개
    🟡 관찰 (최근2년상승) : 54개
    🔴 위험 (최근가속+임계초과) : 49개
    🟠 주의 (최근급등) : 44개
    🔴 위험 (3년연속상승+임계초과) : 38개
    🔴 위험 (최근2년상승+임계초과) : 30개
    🔴 위험 (최근급등+임계초과) : 29개
    🔴 위험 (임계초과) : 23개
    🟡 관찰 (3년연속상승) : 22개
    🔴 위험 (가속+임계초과) : 18개
    🟡 관찰 (가속) : 15개
    🟠 주의 (최근가속+임계근접) : 11개
    ⚪ 보통 : 11개
    🟠 주의 (가속+임계근접) : 4개
    🟠 주의 (3년연속상승+임계근접) : 2개
    🟠 주의 (최근2년상승+임계근접) : 2개

  [상위 10개 (risk_score 높은 순)]
          회사명  prob_2014  prob_2015  prob_2016  prob_2017  prob_2018  prob_2019  prob_2020  prob_2021  prob_2022  prob_2023  prob_2024  diff_2014_2015  diff_2015_2016  diff_2016_2017  diff_2017_2018  diff_2018_2019  diff_2019_2020  diff_2020_2021  diff_2021_2022  diff_2022_2023  diff_2023_2024  risk_score       _risk_signal
     에이스메탈(주)     0.0

# 검증용

In [18]:
# ============================================================
# 기업 부실 확률 변화(차이값) 분석
# 2014_2024_PD_데이터.csv 기반 (2014~2024, 11개 연도 → 10개 차이값)
# - 별도 기업 필터링 없음 (CSV에 있는 데이터 그대로 사용)
# - 해당 연도 PD 값이 없으면 NaN으로 처리
# ============================================================

import pandas as pd
import numpy as np
import os

PD_SAVE_DIR = r'15번. 우수모델 데이터'
PD_PATH     = os.path.join(PD_SAVE_DIR, "2014_2024_PD_데이터.csv")

YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
ID_COL       = "사업자등록번호"
THRESHOLD    = 0.44

TARGET_YEARS = list(range(2014, 2025))   # 2014 ~ 2024, 총 11개 연도

SAVE_DIR     = r"21번. 기업 PD 변화율"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── CSV 로드 ──────────────────────────────────────────────
pd_df = pd.read_csv(PD_PATH, encoding="utf-8-sig")
pd_df[YEAR_COL] = pd_df[YEAR_COL].astype(int)

print("=" * 65)
print(f"PD 데이터 로드 완료: {len(pd_df)}행")
print(f"연도 범위: {pd_df[YEAR_COL].min()} ~ {pd_df[YEAR_COL].max()}")
print(f"기업 수: {pd_df[ID_COL].nunique()}개")
print("=" * 65)


# ============================================================
# 분석 대상 데이터 (CSV 전체, 2014~2024 연도만 사용)
# ============================================================

analysis_data = pd_df[
    pd_df[YEAR_COL].isin(TARGET_YEARS)
].copy()

print(f"분석 대상 행 수 ({TARGET_YEARS[0]}~{TARGET_YEARS[-1]}): {len(analysis_data)}행")
print(f"분석 대상 기업 수: {analysis_data[ID_COL].nunique()}개")
print("=" * 65)


# ============================================================
# 연도별 피벗
# - 해당 연도에 데이터가 없는 기업은 자동으로 NaN
# ============================================================

pivot = analysis_data.pivot_table(
    index=[ID_COL, COMPANY_COL],
    columns=YEAR_COL,
    values="y_prob"
).reset_index()
pivot.columns.name = None

rename_map = {yr: f"prob_{yr}" for yr in TARGET_YEARS}
pivot = pivot.rename(columns=rename_map)

prob_cols = [f"prob_{yr}" for yr in TARGET_YEARS]

# 혹시 일부 연도 컬럼 자체가 존재하지 않을 경우(해당 연도 데이터가 전혀 없음) 대비
for col in prob_cols:
    if col not in pivot.columns:
        pivot[col] = np.nan


# ============================================================
# ★ 차이값 계산 (현재 - 이전, 단위: 확률 그대로)
# 양수 = 부실확률 상승, 음수 = 부실확률 하락
# 둘 중 하나라도 값이 없으면 NaN
# 2014~2024 → diff_2014_2015 ... diff_2023_2024 (총 10개)
# ============================================================

diff_cols = []
for prev_yr, curr_yr in zip(TARGET_YEARS[:-1], TARGET_YEARS[1:]):
    col_name = f"diff_{prev_yr}_{curr_yr}"
    pivot[col_name] = (pivot[f"prob_{curr_yr}"] - pivot[f"prob_{prev_yr}"]).round(4)
    diff_cols.append(col_name)


# ============================================================
# 위험 신호 분류 (최근 3개 차이값 기준)
# 가속: diff 자체가 점점 커지는 것 (d_a < d_b < d_c)
# - 값 중 NaN이 있으면 추세 조건은 모두 False 처리됨
#   (NaN 비교는 항상 False) → "데이터부족" 신호로 분류
# ============================================================

LAST_YEAR   = TARGET_YEARS[-1]
last3_diffs = diff_cols[-3:]   # 예: ['diff_2021_2022', 'diff_2022_2023', 'diff_2023_2024']
last2_diffs = diff_cols[-2:]   # 예: ['diff_2022_2023', 'diff_2023_2024']

def risk_signal(row):
    p_last = row[f"prob_{LAST_YEAR}"]

    # 최종 연도 PD 자체가 없으면 평가 불가
    if pd.isna(p_last):
        return "⚪ 데이터없음(최종연도)"

    d_a, d_b, d_c = (row[c] for c in last3_diffs)   # 최근 3개 차이값
    d_b2, d_c2    = (row[c] for c in last2_diffs)   # 최근 2개 차이값 (d_b == d_b2, d_c == d_c2)

    # 최근 3년 연속 상승 + 가속 (diff 자체도 점점 커짐)
    accel_3 = (
        d_a > 0 and d_b > 0 and d_c > 0 and
        d_a < d_b < d_c
    )
    # 최근 3년 연속 상승
    rising_3 = d_a > 0 and d_b > 0 and d_c > 0

    # 최근 2년 상승 + 가속
    accel_2  = d_b2 > 0 and d_c2 > 0 and d_b2 < d_c2

    # 최근 2년 상승
    rising_2 = d_b2 > 0 and d_c2 > 0

    if accel_3:
        if p_last >= THRESHOLD:
            return "🔴 위험 (가속+임계초과)"
        elif p_last >= THRESHOLD * 0.7:
            return "🟠 주의 (가속+임계근접)"
        else:
            return "🟡 관찰 (가속)"

    if rising_3:
        if p_last >= THRESHOLD:
            return "🔴 위험 (3년연속상승+임계초과)"
        elif p_last >= THRESHOLD * 0.7:
            return "🟠 주의 (3년연속상승+임계근접)"
        else:
            return "🟡 관찰 (3년연속상승)"

    if accel_2:
        if p_last >= THRESHOLD:
            return "🔴 위험 (최근가속+임계초과)"
        elif p_last >= THRESHOLD * 0.7:
            return "🟠 주의 (최근가속+임계근접)"
        else:
            return "🟡 관찰 (최근가속)"

    if rising_2:
        if p_last >= THRESHOLD:
            return "🔴 위험 (최근2년상승+임계초과)"
        elif p_last >= THRESHOLD * 0.7:
            return "🟠 주의 (최근2년상승+임계근접)"
        else:
            return "🟡 관찰 (최근2년상승)"

    # 최근 구간만 큰 폭 상승 (0.05 이상)
    if pd.notna(d_c2) and d_c2 > 0.05:
        if p_last >= THRESHOLD:
            return "🔴 위험 (최근급등+임계초과)"
        else:
            return "🟠 주의 (최근급등)"

    # 추세 무관 임계값 초과
    if p_last >= THRESHOLD:
        return "🔴 위험 (임계초과)"

    # 안정
    if p_last < THRESHOLD * 0.5:
        return "🟢 안정"

    return "⚪ 보통"


pivot["_risk_signal"] = pivot.apply(risk_signal, axis=1)


# ============================================================
# 순위 기반 위험 점수 (0~100)
# 최근 연도일수록 가중치 높게 (최근 3개 차이값 + 최종 연도 확률)
# - rank(pct=True)는 NaN을 자동으로 제외하고, NaN 입력은 NaN으로 반환됨
#   → 최근 3개 diff나 최종연도 PD 중 NaN이 있는 행은 risk_score도 NaN
# ============================================================

rank_d_a_col = f"rank_{last3_diffs[0]}"
rank_d_b_col = f"rank_{last3_diffs[1]}"
rank_d_c_col = f"rank_{last3_diffs[2]}"
rank_prob_col = f"rank_prob_{LAST_YEAR}"

pivot[rank_d_a_col]  = pivot[last3_diffs[0]].rank(pct=True) * 100
pivot[rank_d_b_col]  = pivot[last3_diffs[1]].rank(pct=True) * 100
pivot[rank_d_c_col]  = pivot[last3_diffs[2]].rank(pct=True) * 100
pivot[rank_prob_col] = pivot[f"prob_{LAST_YEAR}"].rank(pct=True) * 100

# 추세 점수 (가중 합산) + 현재 절대 수준 반영
pivot["risk_score"] = (
    (
        pivot[rank_d_a_col] * 0.2 +
        pivot[rank_d_b_col] * 0.3 +
        pivot[rank_d_c_col] * 0.5
    ) * 0.5 +
    pivot[rank_prob_col] * 0.5
).round(2)


# ============================================================
# 컬럼 정리
# ============================================================

result_df = pivot[[
    ID_COL, COMPANY_COL,
    *prob_cols,
    *diff_cols,
    "risk_score",
    "_risk_signal",
]].copy()

# ── prob 소수점 (NaN은 그대로 유지) ────────────────────────
for col in prob_cols:
    result_df[col] = result_df[col].round(4)

# ── risk_score 소수점 ─────────────────────────────────────
result_df["risk_score"] = result_df["risk_score"].round(2)


# ============================================================
# 정렬 (risk_score 내림차순, NaN은 맨 뒤로)
# ============================================================

result_df = result_df.sort_values(
    "risk_score", ascending=False, na_position="last"
).reset_index(drop=True)


# ============================================================
# 저장 (_risk_signal 제외)
# ============================================================

save_cols        = [c for c in result_df.columns if c != "_risk_signal"]
change_save_path = os.path.join(SAVE_DIR, "2015-2024_기업_부실확률_차이값.csv")
result_df[save_cols].to_csv(
    change_save_path, index=False, encoding="utf-8-sig"
)


# ============================================================
# 요약 출력
# ============================================================

print(f"\n{'='*65}")
print(f"기업 부실 확률 차이값 분석 완료")
print(f"  → {change_save_path}  ({len(result_df)}행)")

print(f"\n  [위험 신호 분포]")
for sig, cnt in result_df["_risk_signal"].value_counts(dropna=False).items():
    print(f"    {sig} : {cnt}개")

print(f"\n  [상위 10개 (risk_score 높은 순)]")
print(result_df[[
    COMPANY_COL,
    *prob_cols,
    *diff_cols,
    "risk_score", "_risk_signal"
]].head(10).to_string(index=False))

print("=" * 65)

PD 데이터 로드 완료: 35180행
연도 범위: 2014 ~ 2024
기업 수: 5684개
분석 대상 행 수 (2014~2024): 35180행
분석 대상 기업 수: 5684개

기업 부실 확률 차이값 분석 완료
  → 21번. 기업 PD 변화율\2015-2024_기업_부실확률_차이값.csv  (5684행)

  [위험 신호 분포]
    🟢 안정 : 2062개
    ⚪ 데이터없음(최종연도) : 1975개
    🟡 관찰 (최근가속) : 443개
    🔴 위험 (최근가속+임계초과) : 249개
    🟡 관찰 (최근2년상승) : 165개
    🟠 주의 (최근급등) : 145개
    🔴 위험 (3년연속상승+임계초과) : 133개
    🔴 위험 (임계초과) : 90개
    🔴 위험 (최근급등+임계초과) : 85개
    🔴 위험 (최근2년상승+임계초과) : 67개
    🔴 위험 (가속+임계초과) : 60개
    🟡 관찰 (3년연속상승) : 58개
    🟡 관찰 (가속) : 45개
    🟠 주의 (최근가속+임계근접) : 41개
    ⚪ 보통 : 40개
    🟠 주의 (최근2년상승+임계근접) : 10개
    🟠 주의 (3년연속상승+임계근접) : 8개
    🟠 주의 (가속+임계근접) : 8개

  [상위 10개 (risk_score 높은 순)]
          회사명  prob_2014  prob_2015  prob_2016  prob_2017  prob_2018  prob_2019  prob_2020  prob_2021  prob_2022  prob_2023  prob_2024  diff_2014_2015  diff_2015_2016  diff_2016_2017  diff_2017_2018  diff_2018_2019  diff_2019_2020  diff_2020_2021  diff_2021_2022  diff_2022_2023  diff_2023_2024  risk_score       _risk_signal
    (주)비전오토모빌 